In [1]:
import boto3, yaml
from pathlib import Path

with open("conf/env.yml")  as f: env  = yaml.safe_load(f)
with open("conf/meta.yml") as f: meta = yaml.safe_load(f)

REGION       = env['region']
MODEL_BUCKET = env['s3']['model_bucket']
ENV          = env['env']
USER_ID      = meta['user_id']
PROJECT      = meta['project']
EXPERIMENT   = meta['experiment']
RUN_ID       = "20260305_lightgbm_baseline_b307c3ad"

print(f"Region      : {REGION}")
print(f"Model Bucket: s3://{MODEL_BUCKET}/")
print(f"Run ID      : {RUN_ID}")

Region      : us-east-1
Model Bucket: s3://gs-retail-awesome-model-us-east-1/
Run ID      : 20260305_lightgbm_baseline_b307c3ad


In [2]:
s3 = boto3.client('s3', region_name=REGION)

def ensure_bucket(bucket):
    try:
        s3.head_bucket(Bucket=bucket)
        print(f"  ✅ 이미 존재: {bucket}")
    except:
        cfg = {} if REGION == 'us-east-1' else {'CreateBucketConfiguration': {'LocationConstraint': REGION}}
        s3.create_bucket(Bucket=bucket, **cfg)
        print(f"  🪣 버킷 생성: {bucket}")

def upload_dir(local_dir, bucket, s3_prefix):
    for p in Path(local_dir).rglob("*"):
        if p.is_file():
            key = s3_prefix + str(p.relative_to(local_dir)).replace("\\", "/")
            s3.upload_file(str(p), bucket, key)
            print(f"  📤 {p.name} → {key}")

In [3]:
model_prefix = f"{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}/{RUN_ID}/"
print(f"model 버킷 시딩 → s3://{MODEL_BUCKET}/{model_prefix}")

ensure_bucket(MODEL_BUCKET)
upload_dir(f"output/{RUN_ID}", MODEL_BUCKET, model_prefix)

model 버킷 시딩 → s3://gs-retail-awesome-model-us-east-1/dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/
  ✅ 이미 존재: gs-retail-awesome-model-us-east-1
  📤 meta.yml → dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/config/meta.yml
  📤 model.yml → dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/config/model.yml
  📤 env.yml → dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/config/env.yml
  📤 run_manifest.yml → dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/metadata/run_manifest.yml
  📤 classification_report.html → dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/reports/classification_report.html
  📤 input_data_ref.yml → dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/data_refs/input_data_ref.yml
  📤 confusion_matrix.png → dev/sean/titani

In [4]:
resp = s3.list_objects_v2(Bucket=MODEL_BUCKET, Prefix=model_prefix)
print(f"\n✅ 업로드된 파일 ({len(resp.get('Contents', []))}개):")
for obj in resp.get('Contents', []):
    print(f"  {obj['Key'].replace(model_prefix, '')}  ({obj['Size']//1024}KB)")

manifest_uri = f"s3://{MODEL_BUCKET}/{model_prefix}metadata/run_manifest.yml"
print(f"\n[수강생 실습 명령어]")
print(f"aws s3 cp {manifest_uri} .")


✅ 업로드된 파일 (14개):
  artifacts/charts/confusion_matrix.png  (41KB)
  artifacts/charts/feature_importance.png  (45KB)
  artifacts/charts/learning_curve.png  (100KB)
  artifacts/charts/roc_curve.png  (62KB)
  artifacts/explainability/feature_impact_summary.png  (45KB)
  artifacts/metrics/model_metrics.yml  (1KB)
  artifacts/model/model_baseline.pkl  (336KB)
  artifacts/model/model_baseline.txt  (335KB)
  config/env.yml  (1KB)
  config/meta.yml  (1KB)
  config/model.yml  (3KB)
  data_refs/input_data_ref.yml  (1KB)
  metadata/run_manifest.yml  (1KB)
  reports/classification_report.html  (4KB)

[수강생 실습 명령어]
aws s3 cp s3://gs-retail-awesome-model-us-east-1/dev/sean/titanic-survival-prediction/baseline-v1/20260305_lightgbm_baseline_b307c3ad/metadata/run_manifest.yml .
